# MOVIE RECOMMENDER SYSTEM

## Project Introduction
This project aims to develop a machine learning model that predicts movie ratings based on user–movie interactions and movie-related features. The goal is to generate accurate rating predictions for unseen user–movie pairs using the provided training and test datasets.
The project implements a hybrid recommendation approach that combines two models: Singular Value Decomposition (SVD), which uses collaborative filtering to learn patterns from historical user ratings, and LightGBM, which predicts ratings using engineered movie features derived from tag-genome relevance scores and available IMDb metadata.
The models are evaluated using Root Mean Squared Error (RMSE), where lower values indicate better prediction accuracy. A validation dataset is used to compare model performance and determine the optimal weight for combining their predictions. Finally, both models are retrained using the full training dataset, and their predictions are combined to generate the final submission.csv file.

## Project Workflow
The project is organized into the following major steps:

1.	Environment Setup and Data Loading

3.	Data Splitting and Baseline Modeling 
4.	Movie Feature Engineering
5.	LightGBM Model Training 
6.	Ensemble Optimization 
7.	Final Model Training 
8.	Prediction and Submission Generation 

## Step 1: Set up the environment and load the data

In [2]:
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from surprise import Dataset, Reader, SVD
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

# Suppress non-critical warnings for cleaner notebook output.
warnings.filterwarnings("ignore")

# Set a fixed random seed for reproducible results.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# Loading data
train = pd.read_csv(
    "train.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32"
    }
)

test = pd.read_csv(
    "test.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32"
    }
)

In [3]:
print("Training shape:", train.shape)
print("Test shape:", test.shape)

Training shape: (10000038, 4)
Test shape: (5000019, 2)


In [4]:
display(train.head())
display(test.head())

,userId,movieId,rating,timestamp
0,5163,57669,4.0,1518349992
1,106343,5,4.5,1206238739
2,146790,5459,5.0,1076215539
3,106362,32296,2.0,1423042565
4,9041,366,3.0,833375837


,userId,movieId
0,1,2011
1,1,4144
2,1,5767
3,1,6711
4,1,7318


In [5]:
# Check for missing values in the training data.
print("\nMissing values in training data:")
print(train.isnull().sum())


Missing values in training data:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [6]:
# Examine the distribution of known ratings.
print("\nRating statistics:")
print(train["rating"].describe())


Rating statistics:
count    1.000004e+07
mean     3.533395e+00
std      1.036790e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64


## Step 2: Split the data and train the baseline model

In [7]:
# Splitting the data
train_part, val_part = train_test_split(
    train,
    test_size=0.10,
    random_state=RANDOM_STATE
)

# Reset indices for easier handling.
train_part = train_part.reset_index(drop=True)
val_part = val_part.reset_index(drop=True)

print("Training rows:", len(train_part))
print("Validation rows:", len(val_part))

Training rows: 9000034
Validation rows: 1000004


In [9]:
reader = Reader(rating_scale=(0.5, 5.0))

# Candidate configurations to test
configs = [
    (50, 30, 0.005, 0.05),
    (100, 40, 0.005, 0.05),
    (150, 40, 0.005, 0.05),
    (200, 40, 0.005, 0.05),
    (100, 50, 0.005, 0.02),
    (150, 50, 0.005, 0.02),
    (200, 50, 0.003, 0.05),
    (150, 40, 0.003, 0.10),
]

# Prepare the training dataset once
data = Dataset.load_from_df(
    train_part[["userId", "movieId", "rating"]],
    reader
)
full_trainset = data.build_full_trainset()

# Track the best configuration
results = []
best_rmse = float("inf")
best_svd = None
best_params = None
best_predictions = None

# Train and evaluate each configuration
for factors, epochs, lr, reg in configs:

    model = SVD(
        n_factors=factors,
        n_epochs=epochs,
        lr_all=lr,
        reg_all=reg,
        random_state=RANDOM_STATE
    )

    model.fit(full_trainset)

    predictions = np.array([
        model.predict(int(user), int(movie)).est
        for user, movie in zip(
            val_part["userId"],
            val_part["movieId"]
        )
    ])
    rmse = np.sqrt(mean_squared_error(
    val_part["rating"].values,
    predictions
    ))

    results.append({
        "n_factors": factors,
        "n_epochs": epochs,
        "learning_rate": lr,
        "regularization": reg,
        "RMSE": rmse
    })

    print(
        f"Factors={factors}, Epochs={epochs}, "
        f"LR={lr}, Reg={reg} -> RMSE={rmse:.6f}"
    )

    if rmse < best_rmse:
        best_rmse = rmse
        best_svd = model
        best_predictions = predictions
        best_params = {
            "n_factors": factors,
            "n_epochs": epochs,
            "lr_all": lr,
            "reg_all": reg
        }

# Display results, best first
results_df = pd.DataFrame(results).sort_values("RMSE")

print("\nBest SVD parameters:", best_params)
print(f"Best validation RMSE: {best_rmse:.6f}")

display(results_df)

# Keep these variables for the next notebook steps
svd_model = best_svd
svd_predictions = best_predictions
svd_rmse = best_rmse

Factors=50, Epochs=30, LR=0.005, Reg=0.05 -> RMSE=0.824702
Factors=100, Epochs=40, LR=0.005, Reg=0.05 -> RMSE=0.813309
Factors=150, Epochs=40, LR=0.005, Reg=0.05 -> RMSE=0.812590
Factors=200, Epochs=40, LR=0.005, Reg=0.05 -> RMSE=0.812398
Factors=100, Epochs=50, LR=0.005, Reg=0.02 -> RMSE=0.840443
Factors=150, Epochs=50, LR=0.005, Reg=0.02 -> RMSE=0.839537
Factors=200, Epochs=50, LR=0.003, Reg=0.05 -> RMSE=0.819035
Factors=150, Epochs=40, LR=0.003, Reg=0.1 -> RMSE=0.853060

Best SVD parameters: {'n_factors': 200, 'n_epochs': 40, 'lr_all': 0.005, 'reg_all': 0.05}
Best validation RMSE: 0.812398


,n_factors,n_epochs,learning_rate,regularization,RMSE
3,200,40,0.005,0.05,0.812398
2,150,40,0.005,0.05,0.812590
1,100,40,0.005,0.05,0.813309
6,200,50,0.003,0.05,0.819035
0,50,30,0.005,0.05,0.824702
5,150,50,0.005,0.02,0.839537
4,100,50,0.005,0.02,0.840443
7,150,40,0.003,0.10,0.853060


## Step 3: Engineer movie features

In [3]:
# Select informative tag-genome features
chunk_size = 1_000_000

# Running totals allow statistics to be calculated across chunks.
tag_sum = {}
tag_sq_sum = {}
tag_count = {}


for chunk in pd.read_csv(
    "genome_scores.csv",
    usecols=["tagId", "relevance"],
    dtype={
        "tagId": "int32",
        "relevance": "float32"
    },
    chunksize=chunk_size
):

    # Calculate per-tag sums and observation counts.
    grouped = chunk.groupby("tagId")["relevance"].agg(
        ["sum", "count"]
    )

    # Calculate squared relevance totals for variance.
    squared = (
        chunk.assign(
            squared=chunk["relevance"] ** 2
        )
        .groupby("tagId")["squared"]
        .sum()
    )

    # Accumulate statistics across chunks.
    for tag_id, row in grouped.iterrows():

        tag_id = int(tag_id)

        tag_sum[tag_id] = (
            tag_sum.get(tag_id, 0.0)
            + float(row["sum"])
        )

        tag_sq_sum[tag_id] = (
            tag_sq_sum.get(tag_id, 0.0)
            + float(squared.loc[tag_id])
        )

        tag_count[tag_id] = (
            tag_count.get(tag_id, 0)
            + int(row["count"])
        )


# Calculate global mean and variance for each tag.
tag_ids = np.array(list(tag_sum.keys()))

counts = np.array(
    [tag_count[tag_id] for tag_id in tag_ids],
    dtype=np.float64
)

means = np.array([
    tag_sum[tag_id] for tag_id in tag_ids
]) / counts

variances = (
    np.array([
        tag_sq_sum[tag_id] for tag_id in tag_ids
    ]) / counts
    - means ** 2
)


# Rank tags by variance, from highest to lowest.
tag_variance = pd.DataFrame({
    "tagId": tag_ids,
    "variance": variances
}).sort_values(
    "variance",
    ascending=False
)


# Keep the 80 highest-variance tags.
selected_tags = (
    tag_variance.head(80)["tagId"].tolist()
)

print("Selected tags:", len(selected_tags))
print("Top 10 tags by variance:")

display(tag_variance.head(10))


# Release temporary dictionaries to reduce memory use.
del tag_sum, tag_sq_sum, tag_count
gc.collect()


Selected tags: 80
Top 10 tags by variance:


,tagId,variance
229,230,0.080184
18,19,0.063892
845,846,0.058287
521,522,0.057670
1019,1020,0.057518
539,540,0.055903
556,557,0.055075
269,270,0.054883
322,323,0.054406
1083,1084,0.054314


20

In [4]:
# Load and transform the selected genome features
genome_chunks = []
for chunk in pd.read_csv(
    "genome_scores.csv",
    usecols=["movieId", "tagId", "relevance"],
    dtype={
        "movieId": "int32",
        "tagId": "int32",
        "relevance": "float32"
    },
    chunksize=chunk_size
):
    # Retain only the selected tag IDs.
    filtered = chunk[
        chunk["tagId"].isin(selected_tags)
    ]

    if not filtered.empty:
        genome_chunks.append(filtered)

# Combine filtered chunks.
genome_selected = pd.concat(
    genome_chunks,
    ignore_index=True
)
del genome_chunks
gc.collect()

# Pivot to one row per movie and one column per tag.
genome_features = genome_selected.pivot(
    index="movieId",
    columns="tagId",
    values="relevance"
)

# Give the feature columns descriptive names.
genome_features.columns = [
    f"genome_{tag}"
    for tag in genome_features.columns
]

# Use memory-efficient numeric types and fill missing scores.
genome_features = (
    genome_features
    .astype("float32")
    .fillna(0)
)
genome_features = genome_features.reset_index()
print("Genome feature shape:", genome_features.shape)
display(genome_features.head())

# Release intermediate data.
del genome_selected
gc.collect()

Genome feature shape: (13816, 81)


,movieId,genome_19,genome_22,genome_30,genome_64,genome_107,genome_113,genome_140,genome_143,genome_192,...,genome_1008,genome_1014,genome_1020,genome_1025,genome_1084,genome_1085,genome_1090,genome_1091,genome_1092,genome_1104
0,1,0.66250,0.28600,0.67625,0.98425,0.1635,0.50100,0.11525,0.16625,0.31225,...,0.53875,0.1380,0.32350,0.28700,0.23375,0.17300,0.56375,0.3150,0.67325,0.42700
1,2,0.64025,0.48450,0.12675,0.14600,0.8175,0.64675,0.02150,0.06550,0.14425,...,0.14850,0.2475,0.12900,0.15850,0.18475,0.11300,0.38150,0.2250,0.21700,0.38650
2,3,0.16025,0.19375,0.09550,0.03750,0.1170,0.15825,0.02475,0.08500,0.13950,...,0.16975,0.1955,0.11775,0.09350,0.13450,0.09825,0.10725,0.1960,0.09850,0.24350
3,4,0.14700,0.46175,0.13150,0.05000,0.5360,0.36600,0.02575,0.06350,0.13050,...,0.18200,0.2220,0.06675,0.05275,0.15075,0.08650,0.10750,0.2115,0.11625,0.21975
4,5,0.15575,0.19800,0.11875,0.03550,0.1015,0.16225,0.02475,0.05350,0.09275,...,0.19300,0.1715,0.09475,0.07225,0.09225,0.07075,0.11925,0.1850,0.11875,0.29350


0